In [1]:
%cd ../../

/path/to/repo/my-coles-gnn-experimetns/scenario_age_pred


/path/to/repo/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from collections import defaultdict
import os

BATCH_SIZE = 40
MAX_EPOCHES_OPTIONS = range(10, 160, 10) 
SPLIT_COUNT = 2



expected_experiments = []

for train_epochs in MAX_EPOCHES_OPTIONS:
    experiment_name = f'coles__batch_size_{BATCH_SIZE}__split_count_{SPLIT_COUNT}__{train_epochs}_epoches'
    expected_experiments.append(experiment_name)


In [3]:
expected_experiments

['coles__batch_size_40__split_count_2__10_epoches',
 'coles__batch_size_40__split_count_2__20_epoches',
 'coles__batch_size_40__split_count_2__30_epoches',
 'coles__batch_size_40__split_count_2__40_epoches',
 'coles__batch_size_40__split_count_2__50_epoches',
 'coles__batch_size_40__split_count_2__60_epoches',
 'coles__batch_size_40__split_count_2__70_epoches',
 'coles__batch_size_40__split_count_2__80_epoches',
 'coles__batch_size_40__split_count_2__90_epoches',
 'coles__batch_size_40__split_count_2__100_epoches',
 'coles__batch_size_40__split_count_2__110_epoches',
 'coles__batch_size_40__split_count_2__120_epoches',
 'coles__batch_size_40__split_count_2__130_epoches',
 'coles__batch_size_40__split_count_2__140_epoches',
 'coles__batch_size_40__split_count_2__150_epoches']

In [4]:
HYDRA_OUTPUTS_PATH ='hydra_outputs/MERGED_SUPERCOMPUTER_AND_S_COMPUTER_HYDRA'

In [5]:
all_hydra_outputs = os.listdir(HYDRA_OUTPUTS_PATH)
MAX_EPOCHES = 150
full_train_experiment_name = f'coles__batch_size_{BATCH_SIZE}__split_count_{SPLIT_COUNT}__{MAX_EPOCHES}_epoches'
full_train_experiment_name in all_hydra_outputs

True

In [6]:
import sys, os; sys.path.append(os.path.abspath( '..'))

In [7]:
from omegaconf import OmegaConf 

In [8]:
import sys
import os

sys.path.append(os.path.dirname(os.getcwd()))

In [9]:
from ptls_extension_2024_research.latex_table_creation.latex_table_creation import create_latex_table, get_metrics
from ptls_extension_2024_research.latex_table_creation.experiment_dicts_list_modifiers import bolden_top_k, sort_by_col
from ptls_extension_2024_research.latex_table_creation.prefix_map import get_idxs_where_all_metrics_superpass, prefix_map_from_idx_lst

/path/to/repo/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
with open("results/scenario_age_coles_baseline.txt") as f:
    report_file_content = f.read()

In [11]:
from typing import Optional
import re


from ptls_extension_2024_research.latex_table_creation.hyperparam_getters import get_batchsize_from_config, get_split_count_from_config


# Some information is present in name only, everything else is 
# suposed to be retrived from the config (yaml read wit OmegaConf).
# For a uniformal interface both config and experiment_name are always passed to hyperparam getters.





# def get_gnn_loss_alpha(config, experiment_name) -> Optional[float]:
#     matches = re.search(r'wl-(\d+\.\d+)', experiment_name)
#     if matches is None:
#         return None
#     return float(matches.group(1))


# def get_gnn_task(config, experiment_name) -> str:
#     if 'GRACE' in experiment_name:
#         return 'GRACE'
#     return 'Link and weight prediction'


# def get_has_residual(config, experiment_name) -> str:
#     has_residual_str = re.search(r'residual-(true|True|False)', experiment_name).group(1)
#     if  has_residual_str in ['true', 'True']:
#         return True
#     return False

# def get_weight_decay(config, experiment_name):
#     return re.search(r'weight_decay-(\d*\.?\d+)', experiment_name).group(1)


def get_train_epoches(config, experiment_name) -> int:
    # Look for 'epoches_<number>' but exclude 'pretrain_epoches_<number>'
    matches = re.search(r'(\d+)_epoches', experiment_name)
    return int(matches.group(1))





In [12]:

hyperparams_to_getters = {
    r"\textbf{Batch size}": get_batchsize_from_config,
    r"\textbf{Split count}": get_split_count_from_config,
    r"\textbf{Train epoches}": get_train_epoches,

}

In [28]:
experiment_dicts_list = []

for experiment_name in expected_experiments:
    hydra_config_path = os.path.join(HYDRA_OUTPUTS_PATH, experiment_name, '.hydra', 'config.yaml')
    hydra_config_path = re.sub(r'(\d+)_epoches', '150_epoches', hydra_config_path)
    config = OmegaConf.load(hydra_config_path)
    hyperparams = {k: v(config, experiment_name) for k, v in hyperparams_to_getters.items()}
    metrics = get_metrics(report_file_content, experiment_name, {'accuracy': r"\textbf{ACC test}"})
    if metrics is not None:
        experiment_dicts_list.append({**hyperparams, **metrics})

Could not find experiment coles__batch_size_40__split_count_2__140_epoches in file content


In [29]:
WEIGHTS_TYPE = 'type 2'

# experiment_dicts_list stores raw values

# experiment_dicts_list stores raw values, list is sorted
experiment_dicts_list = sort_by_col(experiment_dicts_list, r"\textbf{ACC test}")
# prefix_map = prefix_map_from_idx_lst(get_idxs_where_all_metrics_superpass(experiment_dicts_list, COLES_METRICS), "\n\\rowcolor{gray!50}\n")

# experiment_dicts_list stores strings
experiment_dicts_of_str_list = bolden_top_k(experiment_dicts_list, 3, [r"\textbf{ACC test}"])




EXPERIMENT_NAME = r'\makecell{Coles only with bpr}'

experiment_data = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: experiment_dicts_of_str_list
    }
}



In [30]:
hyperparameters = hyperparameter_header_strs = list(experiment_dicts_list[0].keys())

caption = "Coles baseline. Scenario age"

row_prefix_dict = None

# row_prefix_dict = {
#     EXPERIMENT_NAME: {
#         WEIGHTS_TYPE: prefix_map
#     }
# }

latex_table = create_latex_table(experiment_data, hyperparameters, hyperparameter_header_strs, caption, row_prefix_dict)

print(latex_table)

\begin{table*}[h!]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|c|}
\hline
\textbf{Model} & \textbf{Type of weights} & \textbf{Batch size} & \textbf{Split count} & \textbf{Train epoches} & \textbf{ACC test}\\
\hline
\multirow{14}{*}{\makecell{Coles only with bpr}} &\multirow{14}{*}{type 2} & 40 & 2 & 150 & \textbf{0.637} \\ \cline{3-6} 
& &40 & 2 & 90 & \textbf{0.635} \\ \cline{3-6} 
& &40 & 2 & 100 & \textbf{0.634} \\ \cline{3-6} 
& &40 & 2 & 120 & 0.634 \\ \cline{3-6} 
& &40 & 2 & 110 & 0.633 \\ \cline{3-6} 
& &40 & 2 & 70 & 0.631 \\ \cline{3-6} 
& &40 & 2 & 80 & 0.631 \\ \cline{3-6} 
& &40 & 2 & 130 & 0.631 \\ \cline{3-6} 
& &40 & 2 & 40 & 0.63 \\ \cline{3-6} 
& &40 & 2 & 60 & 0.629 \\ \cline{3-6} 
& &40 & 2 & 50 & 0.628 \\ \cline{3-6} 
& &40 & 2 & 20 & 0.625 \\ \cline{3-6} 
& &40 & 2 & 30 & 0.625 \\ \cline{3-6} 
& &40 & 2 & 10 & 0.613 \\ \cline{2-6} 
\hline
\end{tabular}
}
\caption{Coles baseline. Scenario age}
\label{tab:results}
\end{table*}
